# LeetCode 458 — Poor Pigs bằng Python (Hard)

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def poorPigs(self, buckets: int, minutesToDie: int, minutesToTest: int) -> int:

```

Đây là một bài "Hard" rất đặc biệt: **code nộp bài chỉ dài 5–6 dòng**, nhưng để *hiểu vì sao* 5 dòng đó đúng thì cần một chuỗi suy luận đẹp về **mã hóa thông tin**. Ta sẽ đi chậm, từ cách mà người mới tự nghĩ ra, lên dần tới lời giải tối ưu.

Mục tiêu của bài học:

1. **Hiểu đúng luật chơi**: "lượt thử" (round) là gì, vì sao số lượt là `T = minutesToTest // minutesToDie`, và **viết luật chơi thành một bộ mô phỏng** (simulator) bằng Python để tự kiểm tra mọi chiến lược.
2. Học **kiến thức nền từ con số 0**: quy tắc nhân khi đếm, `itertools.product`, **hệ nhị phân** (bit), **hệ cơ số k bất kỳ** (chuyển số ↔ chữ số bằng `%` và `//`), **nguyên lý chuồng bồ câu**.
3. Thử các **cách ngây thơ** (mỗi con lợn thử một xô, tìm kiếm nhị phân tuần tự) và đo xem chúng **tốn lợn / tốn thời gian** tới mức nào.
4. **Cách 1 — một lượt**: mỗi con lợn là **một bit** ⇒ `x` con lợn phân biệt được `2^x` xô.
5. **Cách 2 — nhiều lượt (cách chính)**: mỗi con lợn có `T + 1` **trạng thái** (chết sau lượt 1, 2, …, T, hoặc sống) ⇒ cần `x` nhỏ nhất với `(T + 1)^x ≥ buckets`.
6. **Cách 3 — mô phỏng chiến lược thật**: gán mỗi xô một mã cơ số `T + 1`, cho lợn uống theo mã, rồi **giải mã** xô độc từ thời điểm lợn chết — chạy kiểm chứng cho **mọi** xô.
7. **Cách 4 — tính `x` an toàn**: vì sao `math.ceil(math.log(...))` có thể **sai** (demo cụ thể `buckets = 125`), và cách dùng vòng lặp số nguyên để luôn đúng.
8. Chứng minh **vì sao không thể dùng ít lợn hơn**, so sánh các cách, lỗi thường gặp, nhận dạng pattern "phân biệt N khả năng", bài tập có đáp án.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Có `buckets` xô chất lỏng, trong đó **đúng một xô có độc**. Để tìm ra xô nào có độc, bạn cho một số con lợn (tội nghiệp) uống chất lỏng để xem chúng có chết hay không. Không may, bạn chỉ có `minutesToTest` phút để xác định xô nào có độc.
>
> Bạn có thể cho lợn uống theo các bước sau:
>
> 1. Chọn một số con lợn **còn sống** để cho uống.
> 2. Với mỗi con lợn, chọn những xô nào nó sẽ uống. Con lợn uống **tất cả các xô đã chọn cùng một lúc** và việc uống **không tốn thời gian**. Mỗi con lợn có thể uống từ bao nhiêu xô cũng được, và mỗi xô có thể cho bao nhiêu con lợn uống cũng được.
> 3. Chờ `minutesToDie` phút. Trong khoảng thời gian này bạn **không được cho con lợn nào khác uống**.
> 4. Sau khi `minutesToDie` phút trôi qua, mọi con lợn đã uống xô có độc sẽ chết, những con còn lại sống.
> 5. Lặp lại quy trình này cho tới khi hết thời gian.
>
> Cho `buckets`, `minutesToDie` và `minutesToTest`, hãy trả về **số con lợn ít nhất** cần dùng để xác định được xô có độc trong thời gian cho phép.

**Example 1**

```text
Input: buckets = 4, minutesToDie = 15, minutesToTest = 15
Output: 2
Explanation: Ta có thể xác định xô độc như sau:
Tại phút 0, cho con lợn thứ nhất uống xô 1 và 2, cho con lợn thứ hai uống xô 2 và 3.
Tại phút 15, có 4 khả năng:
- Chỉ con thứ nhất chết  -> xô 1 có độc.
- Chỉ con thứ hai chết   -> xô 3 có độc.
- Cả hai con cùng chết   -> xô 2 có độc.
- Không con nào chết     -> xô 4 có độc.
```

**Example 2**

```text
Input: buckets = 4, minutesToDie = 15, minutesToTest = 30
Output: 2
Explanation: Ta có thể xác định xô độc như sau:
Tại phút 0, cho con lợn thứ nhất uống xô 1, con thứ hai uống xô 2.
Tại phút 15, có 2 khả năng:
- Nếu một trong hai con chết, xô độc là xô mà con đó đã uống.
- Nếu không con nào chết, cho con thứ nhất uống xô 3, con thứ hai uống xô 4.
Tại phút 30, chắc chắn một trong hai con chết, và xô độc là xô mà con đó đã uống.
```

(Bản đề cũ của LeetCode còn có ví dụ kinh điển `buckets = 1000, minutesToDie = 15, minutesToTest = 60` → đáp án `5`. Ta sẽ dùng ví dụ này nhiều lần.)

### Constraints (ràng buộc) và ý nghĩa

```text
1 <= buckets <= 1000
1 <= minutesToDie <= minutesToTest <= 100
```

Đọc ràng buộc như một người giải bài:

- `buckets` có thể bằng `1`: khi chỉ có một xô thì **xô đó chắc chắn có độc** — không cần con lợn nào, đáp án là `0`. Đây là edge case (trường hợp biên) dễ quên nhất.
- `minutesToDie <= minutesToTest`: luôn có **ít nhất một lượt thử**.
- Các số đều nhỏ (≤ 1000, ≤ 100). Đáp án lớn nhất xảy ra khi chỉ có 1 lượt và 1000 xô: cần `10` con lợn (vì `2^10 = 1024 ≥ 1000`). Nên thuật toán nào cũng chạy nhanh — cái khó của bài **không nằm ở tốc độ**, mà ở chỗ **nghĩ ra được** công thức và **chứng minh** nó đúng.
- Đề hỏi **số lợn ít nhất**, không hỏi chiến lược. Nhưng để tin đáp án, ta sẽ tự xây chiến lược và mô phỏng nó.

## 2. Hiểu luật chơi: "lượt thử" là gì?

Hãy hình dung trục thời gian với `minutesToDie = 15`, `minutesToTest = 60`:

```text
phút:   0        15        30        45        60
        |--------|---------|---------|---------|
        cho uống  xem kết   xem kết   xem kết   xem kết
        lượt 1    quả lượt1 quả lượt2 quả lượt3 quả lượt4
                  cho uống  cho uống  cho uống
                  lượt 2    lượt 3    lượt 4
```

- Mỗi **lượt thử** (round): cho lợn uống → chờ đúng `minutesToDie` phút → nhìn xem con nào chết.
- Trong lúc chờ **không được cho uống thêm** (bước 3 của đề), nên các lượt **nối tiếp nhau**, không chồng lên nhau.
- Số lượt tối đa là phép **chia lấy phần nguyên**:

```python
T = minutesToTest // minutesToDie
```

Ví dụ `minutesToDie = 15`, `minutesToTest = 40`: `40 // 15 = 2` lượt. 10 phút còn dư **vô dụng**, vì cho uống ở phút 30 thì phải tới phút 45 mới biết kết quả — đã quá hạn.

Toán tử `//` trong Python là **chia lấy phần nguyên** (floor division): `40 // 15 = 2`, khác với `/` là chia thực: `40 / 15 = 2.666...`.

In [1]:
pairs = [(15, 15), (15, 30), (15, 60), (15, 40), (1, 100), (7, 20)]

for minutesToDie, minutesToTest in pairs:
    T = minutesToTest // minutesToDie
    print(f"minutesToDie={minutesToDie:>3}, minutesToTest={minutesToTest:>3}"
          f" -> T = {minutesToTest} // {minutesToDie} = {T} lượt"
          f"   (còn chia thực: {minutesToTest / minutesToDie:.3f})")

minutesToDie= 15, minutesToTest= 15 -> T = 15 // 15 = 1 lượt   (còn chia thực: 1.000)
minutesToDie= 15, minutesToTest= 30 -> T = 30 // 15 = 2 lượt   (còn chia thực: 2.000)
minutesToDie= 15, minutesToTest= 60 -> T = 60 // 15 = 4 lượt   (còn chia thực: 4.000)
minutesToDie= 15, minutesToTest= 40 -> T = 40 // 15 = 2 lượt   (còn chia thực: 2.667)
minutesToDie=  1, minutesToTest=100 -> T = 100 // 1 = 100 lượt   (còn chia thực: 100.000)
minutesToDie=  7, minutesToTest= 20 -> T = 20 // 7 = 2 lượt   (còn chia thực: 2.857)


### Viết luật chơi thành code: bộ mô phỏng (simulator)

Trước khi nghĩ thuật toán, ta "dịch" luật chơi sang Python. Khi có bộ mô phỏng, **mọi chiến lược ta nghĩ ra đều có thể đem chạy thử**, thay vì tin bằng cảm tính.

Một vài công cụ Python sẽ dùng (giải thích nhanh cho người mới):

- `set` (tập hợp): `{1, 2}` — kiểm tra `x in s` rất nhanh. Dùng để lưu "con lợn này uống những xô nào".
- `dict` (từ điển): `{1: ..., 2: ...}` — ánh xạ **key → value**. Dùng để lưu "lượt `r` → kế hoạch của lượt đó".
- `None`: giá trị "không có gì". Ta dùng `None` để nói "con lợn **không chết**".
- `tuple` (bộ): giống list nhưng không sửa được và **dùng được làm phần tử của set** — rất tiện để so sánh kết quả.

**Cách biểu diễn một kế hoạch (plan):**

```python
plan = {
    1: [ {1, 2}, {2, 3} ],   # lượt 1: lợn 0 uống xô 1, 2; lợn 1 uống xô 2, 3
}
```

`plan[r][j]` = tập các xô mà con lợn `j` uống ở lượt `r` (lượt đánh số từ 1).

**Kết quả của một thí nghiệm** là tuple `death`: `death[j]` = phút mà lợn `j` chết, hoặc `None` nếu nó sống tới cuối.

In [2]:
s = {1, 2}
print("2 in s ->", 2 in s, "| 5 in s ->", 5 in s)

plan_demo = {1: [{1, 2}, {2, 3}]}
print("Lượt 1, lợn 0 uống:", plan_demo[1][0])
print("Lượt 1, lợn 1 uống:", plan_demo[1][1])

outcomes = {(15, None), (None, 15), (15, None)}   # tuple trùng nhau bị gộp
print("Số kết quả khác nhau:", len(outcomes), outcomes)
print()


def simulate_experiment(plan, num_pigs, minutesToDie, minutesToTest, poison):
    # Chạy thí nghiệm theo đúng luật của đề, với xô độc là `poison`.
    # Trả về tuple death: death[j] = phút lợn j chết, hoặc None nếu sống.
    T = minutesToTest // minutesToDie
    assert all(1 <= r <= T for r in plan), "Kế hoạch dùng quá số lượt cho phép!"

    alive = [True] * num_pigs
    death = [None] * num_pigs
    for r in sorted(plan):                       # các lượt nối tiếp nhau
        # Bước 1-2: chỉ lợn CÒN SỐNG mới được cho uống
        drank_poison = [j for j in range(num_pigs)
                        if alive[j] and poison in plan[r][j]]
        # Bước 3-4: chờ minutesToDie phút, lợn uống phải độc sẽ chết
        for j in drank_poison:
            alive[j] = False
            death[j] = r * minutesToDie
    return tuple(death)


def check_plan(plan, num_pigs, bucket_ids, minutesToDie, minutesToTest):
    # Kế hoạch tốt <=> mỗi xô độc cho ra MỘT kết quả khác hẳn các xô còn lại.
    results = {}
    for b in bucket_ids:
        results[b] = simulate_experiment(plan, num_pigs, minutesToDie, minutesToTest, b)
    distinct = len(set(results.values())) == len(bucket_ids)
    return distinct, results

print("Đã định nghĩa simulate_experiment và check_plan")

2 in s -> True | 5 in s -> False
Lượt 1, lợn 0 uống: {1, 2}
Lượt 1, lợn 1 uống: {2, 3}
Số kết quả khác nhau: 2 {(15, None), (None, 15)}

Đã định nghĩa simulate_experiment và check_plan


### Kiểm tra lại Example 1 và Example 2 bằng simulator

**Example 1.**

Kế hoạch của đề (xô đánh số 1..4 như đề): lợn 0 uống `{1, 2}`, lợn 1 uống `{2, 3}`, chỉ 1 lượt.

Bảng tính tay:

| Xô độc | Lợn 0 (uống 1, 2) | Lợn 1 (uống 2, 3) | Kết quả |
|:---:|:---:|:---:|---|
| 1 | chết | sống | `(15, None)` |
| 2 | chết | chết | `(15, 15)` |
| 3 | sống | chết | `(None, 15)` |
| 4 | sống | sống | `(None, None)` |

Bốn xô → bốn kết quả **khác nhau** ⇒ nhìn kết quả là biết ngay xô nào độc. Đó chính là định nghĩa "kế hoạch thành công".

**Example 2.**

Kế hoạch của đề: lượt 1 lợn 0 uống xô 1, lợn 1 uống xô 2; lượt 2 lợn 0 uống xô 3, lợn 1 uống xô 4.

Đề viết "nếu không con nào chết thì…", nghe như phải quyết định tùy kết quả. Nhưng thực ra kế hoạch có thể **viết sẵn từ đầu**: nếu một con đã chết ở lượt 1 thì ta đã biết đáp án, lượt 2 có cho uống hay không cũng không quan trọng (simulator tự bỏ qua lợn đã chết).

Điều đáng chú ý: ở ví dụ này **thời gian chết** cũng mang thông tin. Xô 1 và xô 3 đều làm lợn 0 chết, nhưng chết ở phút **15** hay phút **30** thì khác nhau! Đây là hạt giống của lời giải chính.

In [3]:
plan_ex1 = {1: [{1, 2}, {2, 3}]}
ok, results = check_plan(plan_ex1, 2, [1, 2, 3, 4], 15, 15)
for b, res in results.items():
    print(f"xô độc = {b} -> death = {res}")
print("Mọi kết quả khác nhau?", ok)
print("-" * 40)
plan_ex2 = {1: [{1}, {2}], 2: [{3}, {4}]}
ok, results = check_plan(plan_ex2, 2, [1, 2, 3, 4], 15, 30)
for b, res in results.items():
    print(f"xô độc = {b} -> death = {res}")
print("Mọi kết quả khác nhau?", ok)

xô độc = 1 -> death = (15, None)
xô độc = 2 -> death = (15, 15)
xô độc = 3 -> death = (None, 15)
xô độc = 4 -> death = (None, None)
Mọi kết quả khác nhau? True
----------------------------------------
xô độc = 1 -> death = (15, None)
xô độc = 2 -> death = (None, 15)
xô độc = 3 -> death = (30, None)
xô độc = 4 -> death = (None, 30)
Mọi kết quả khác nhau? True


## 3. Kiến thức nền 1 — Quy tắc nhân và `itertools.product`

**Quy tắc nhân** (product rule): nếu việc thứ nhất có `a` khả năng, việc thứ hai có `b` khả năng, và chúng độc lập, thì cả hai cùng nhau có `a × b` khả năng.

Ví dụ đời thường: 2 cái áo × 3 cái quần = 6 bộ đồ.

Với lợn: mỗi con lợn sau 1 lượt có **2 khả năng** (chết / sống). Hai con lợn ⇒ `2 × 2 = 4` khả năng — đúng bằng số xô trong Example 1! Ba con ⇒ `2 × 2 × 2 = 8`.

Python có sẵn `itertools.product(..., repeat=k)` để **liệt kê** mọi tổ hợp như vậy (tích Descartes):

```python
from itertools import product
list(product(["chết", "sống"], repeat=2))
# [('chết','chết'), ('chết','sống'), ('sống','chết'), ('sống','sống')]
```

In [4]:
from itertools import product

states = ["chết", "sống"]
combos = list(product(states, repeat=2))
for c in combos:
    print(c)
print("Số tổ hợp với 2 con lợn:", len(combos), "= 2 ** 2 =", 2 ** 2)
print("Số tổ hợp với 3 con lợn:", len(list(product(states, repeat=3))), "= 2 ** 3 =", 2 ** 3)

shirts, pants = ["áo A", "áo B"], ["quần 1", "quần 2", "quần 3"]
print("Áo × quần:", len(list(product(shirts, pants))), "bộ")

('chết', 'chết')
('chết', 'sống')
('sống', 'chết')
('sống', 'sống')
Số tổ hợp với 2 con lợn: 4 = 2 ** 2 = 4
Số tổ hợp với 3 con lợn: 8 = 2 ** 3 = 8
Áo × quần: 6 bộ


## 4. Kiến thức nền 2 — Hệ nhị phân (binary)

Ta quen **hệ thập phân** (cơ số 10): `253 = 2·100 + 5·10 + 3·1`. Mỗi chữ số từ `0` tới `9`.

**Hệ nhị phân** (cơ số 2) chỉ có hai chữ số `0` và `1`, gọi là **bit**. Các vị trí có trọng số `1, 2, 4, 8, 16, ...`:

```text
6 = 1·4 + 1·2 + 0·1   ->  viết nhị phân là 110
5 = 1·4 + 0·2 + 1·1   ->  101
```

Cách lấy từng bit của số `n` (từ bit thấp nhất):

- `n % 2` = bit cuối (phần dư khi chia 2).
- `n // 2` = bỏ bit cuối.
- Lặp lại cho tới khi đủ số bit cần.

Ví dụ `n = 6`: `6 % 2 = 0`, `6 // 2 = 3`; `3 % 2 = 1`, `3 // 2 = 1`; `1 % 2 = 1`, `1 // 2 = 0` ⇒ bit từ thấp lên cao: `0, 1, 1` ⇒ đọc ngược `110`.

Python có sẵn `bin(6) = '0b110'` và `format(6, '03b') = '110'` (đủ 3 chữ số, thêm 0 ở đầu nếu thiếu).

**Điểm mấu chốt cho bài này:** với `x` bit, ta viết được đúng `2^x` số khác nhau: `0, 1, ..., 2^x − 1`. Với 3 bit: `000` tới `111`, tức 8 số.

Nếu ta "gắn" mỗi bit cho một con lợn (bit = 1 nghĩa là "con lợn này uống xô đó"), thì mỗi xô có một **mã nhị phân riêng**, và việc lợn nào chết sẽ **đọc ra** được mã của xô độc. Ta sẽ làm điều này ở Cách 1.

In [5]:
def to_bits(n, width):
    bits = []
    for _ in range(width):
        bits.append(n % 2)   # lấy bit cuối
        n //= 2              # bỏ bit cuối
    return bits              # bit thấp nhất đứng đầu list

for n in range(8):
    print(f"n={n}: to_bits={to_bits(n, 3)}  | format(n,'03b')={format(n, '03b')}")
print("bin(6) =", bin(6))

n=0: to_bits=[0, 0, 0]  | format(n,'03b')=000
n=1: to_bits=[1, 0, 0]  | format(n,'03b')=001
n=2: to_bits=[0, 1, 0]  | format(n,'03b')=010
n=3: to_bits=[1, 1, 0]  | format(n,'03b')=011
n=4: to_bits=[0, 0, 1]  | format(n,'03b')=100
n=5: to_bits=[1, 0, 1]  | format(n,'03b')=101
n=6: to_bits=[0, 1, 1]  | format(n,'03b')=110
n=7: to_bits=[1, 1, 1]  | format(n,'03b')=111
bin(6) = 0b110


## 5. Kiến thức nền 3 — Hệ cơ số `k` bất kỳ

Không có gì đặc biệt ở số 2 hay số 10. Với **cơ số `k`**, mỗi chữ số nhận giá trị `0, 1, ..., k − 1`, trọng số các vị trí là `1, k, k², k³, ...`.

Ví dụ cơ số 3: `7 = 2·3 + 1·1` ⇒ chữ số (từ thấp lên cao) là `[1, 2]`, viết thông thường là `21₃`.

**Chuyển số → chữ số** (giống hệt nhị phân, chỉ đổi 2 thành k):

```python
d = n % k    # chữ số thấp nhất
n = n // k   # bỏ chữ số đó
```

Python có hàm `divmod(n, k)` trả về **cả hai** cùng lúc: `divmod(7, 3) = (2, 1)` (thương 2, dư 1).

**Chuyển chữ số → số** (giải mã): `n = d₀·1 + d₁·k + d₂·k² + ...`

Với `x` chữ số cơ số `k`, ta viết được đúng `k^x` số khác nhau (quy tắc nhân: mỗi vị trí có `k` lựa chọn).

In [6]:
def to_digits(n, base, width):
    # Trả về `width` chữ số của n trong cơ số `base`, chữ số thấp nhất đứng đầu.
    digits = []
    for _ in range(width):
        n, d = divmod(n, base)
        digits.append(d)
    return digits


def from_digits(digits, base):
    # Ngược lại: [d0, d1, d2, ...] -> d0 + d1*base + d2*base^2 + ...
    n = 0
    power = 1
    for d in digits:
        n += d * power
        power *= base
    return n


print("divmod(7, 3) =", divmod(7, 3))
print("7 cơ số 3, 2 chữ số  :", to_digits(7, 3, 2))
print("999 cơ số 5, 5 chữ số:", to_digits(999, 5, 5))
print("giải mã lại          :", from_digits(to_digits(999, 5, 5), 5))

for base in [2, 3, 5, 10]:
    for n in range(0, 200, 7):
        assert from_digits(to_digits(n, base, 8), base) == n
print("to_digits / from_digits là cặp ngược nhau: PASS")

divmod(7, 3) = (2, 1)
7 cơ số 3, 2 chữ số  : [1, 2]
999 cơ số 5, 5 chữ số: [4, 4, 4, 2, 1]
giải mã lại          : 999
to_digits / from_digits là cặp ngược nhau: PASS


## 6. Kiến thức nền 4 — Nguyên lý chuồng bồ câu (pigeonhole principle)

> Nếu nhốt **nhiều hơn** `m` con bồ câu vào `m` cái chuồng, thì **chắc chắn** có một chuồng chứa ít nhất 2 con.

Áp dụng: nếu thí nghiệm chỉ có thể cho ra `m` **kết quả khác nhau**, mà có **nhiều hơn** `m` xô, thì chắc chắn có **2 xô khác nhau cho cùng một kết quả**. Khi đó nhìn kết quả ta **không thể** biết xô nào trong hai xô đó có độc ⇒ thí nghiệm thất bại.

Đây là công cụ để chứng minh chiều "**không thể dùng ít lợn hơn**". Thí nghiệm nhỏ: 5 xô, mà chỉ có 4 kết quả có thể (2 con lợn, 1 lượt) — dù gán thế nào cũng có 2 xô trùng.

In [7]:
import random
random.seed(1)

possible_results = list(product(["chết", "sống"], repeat=2))   # 4 kết quả
for attempt in range(3):
    # gán ngẫu nhiên mỗi xô một kết quả (một "chuồng")
    assign = {b: random.choice(possible_results) for b in range(5)}
    seen = {}
    for b, res in assign.items():
        if res in seen:
            print(f"Lần {attempt + 1}: xô {seen[res]} và xô {b} cùng kết quả {res} -> không phân biệt được")
            break
        seen[res] = b

Lần 1: xô 1 và xô 3 cùng kết quả ('chết', 'chết') -> không phân biệt được
Lần 2: xô 0 và xô 1 cùng kết quả ('sống', 'sống') -> không phân biệt được
Lần 3: xô 0 và xô 2 cùng kết quả ('sống', 'sống') -> không phân biệt được


## 7. Kiến thức nền 5 — Lũy thừa tăng rất nhanh

Toán tử `**` là lũy thừa: `2 ** 10 = 1024`. Hãy so sánh:

- Tăng **tuyến tính** (linear): thêm 1 con lợn ⇒ thêm vài xô.
- Tăng **lũy thừa** (exponential): thêm 1 con lợn ⇒ **nhân** số xô lên `k` lần.

Bảng dưới cho thấy vì sao lời giải đúng chỉ cần một nắm lợn: với 5 trạng thái mỗi con, 5 con lợn đã phân biệt được 3125 xô.

In [8]:
print(f"{'x':>2} | {'2*x+1 (tuyến tính)':>18} | {'2**x':>6} | {'3**x':>7} | {'5**x':>8}")
for x in range(0, 11, 2):
    print(f"{x:>2} | {2 * x + 1:>18} | {2 ** x:>6} | {3 ** x:>7} | {5 ** x:>8}")

 x | 2*x+1 (tuyến tính) |   2**x |    3**x |     5**x
 0 |                  1 |      1 |       1 |        1
 2 |                  5 |      4 |       9 |       25
 4 |                  9 |     16 |      81 |      625
 6 |                 13 |     64 |     729 |    15625
 8 |                 17 |    256 |    6561 |   390625
10 |                 21 |   1024 |   59049 |  9765625


## 8. Cách ngây thơ A — "Mỗi con lợn thử một xô"

Người mới thường nghĩ thế này: *cho mỗi con lợn uống đúng một xô, con nào chết thì xô của nó có độc.*

**Một lượt:** với `B` xô, dùng `B − 1` con lợn uống `B − 1` xô; xô cuối không ai uống — nếu không con nào chết thì xô cuối có độc. ⇒ `1000` xô cần `999` con lợn!

**Nhiều lượt:** mỗi lượt, mọi con lợn còn sống thử một xô **mới**. Với `p` con lợn và `T` lượt, thử được `p · T` xô, cộng thêm 1 xô "không ai uống" ⇒ phân biệt được `p · T + 1` xô. Vậy cần

```text
p = ceil((B - 1) / T)
```

Ví dụ `B = 1000, T = 4`: `ceil(999 / 4) = 250` con lợn. So với đáp án đúng là `5`… chênh lệch khổng lồ.

Ta viết code cách này, **kèm kế hoạch thật** và kiểm tra bằng simulator (để chắc nó thực sự đúng, chỉ là tốn).

In [9]:
def naive_one_bucket_each(buckets, minutesToDie, minutesToTest):
    T = minutesToTest // minutesToDie
    pigs = -(-(buckets - 1) // T)        # ceil((buckets-1)/T) bằng số nguyên
    # Kế hoạch: lượt r, lợn j uống đúng 1 xô số (r-1)*pigs + j (nếu còn xô)
    plan = {r: [set() for _ in range(pigs)] for r in range(1, T + 1)}
    for b in range(buckets - 1):         # xô cuối cùng không ai uống
        r, j = divmod(b, pigs)
        plan[r + 1][j].add(b)
    return pigs, plan


for B, die, test in [(4, 15, 15), (4, 15, 30), (10, 15, 45), (1000, 15, 60)]:
    pigs, plan = naive_one_bucket_each(B, die, test)
    ok, _ = check_plan(plan, pigs, range(B), die, test)
    print(f"buckets={B:>4}, T={test // die}: cần {pigs:>3} con lợn, kế hoạch đúng? {ok}")

buckets=   4, T=1: cần   3 con lợn, kế hoạch đúng? True
buckets=   4, T=2: cần   2 con lợn, kế hoạch đúng? True
buckets=  10, T=3: cần   3 con lợn, kế hoạch đúng? True
buckets=1000, T=4: cần 250 con lợn, kế hoạch đúng? True


## 9. Cách ngây thơ B — Tìm kiếm nhị phân tuần tự

Người biết thuật toán sẽ nghĩ tới **binary search** (tìm kiếm nhị phân): cho **một** con lợn uống **một nửa** số xô.

- Nếu nó chết ⇒ xô độc nằm ở nửa đó (và ta mất con lợn này, lượt sau phải dùng con mới).
- Nếu nó sống ⇒ xô độc ở nửa kia (con lợn dùng tiếp được).

Mỗi lượt loại được một nửa ⇒ cần khoảng `log₂(B)` **lượt**. Với `B = 1000` là `10` lượt. Nhưng `minutesToTest = 60, minutesToDie = 15` chỉ cho `4` lượt ⇒ **không kịp giờ**!

Vấn đề: tìm kiếm nhị phân tuần tự làm mọi thứ **nối tiếp**, mỗi lượt chỉ hỏi **một câu hỏi có/không**. Trong khi đề cho phép **nhiều con lợn uống cùng lúc** — ta đang lãng phí khả năng chạy song song.

In [10]:
def sequential_binary_search(buckets, poison):
    lo, hi = 0, buckets - 1          # xô độc nằm trong đoạn [lo, hi]
    rounds = 0
    pigs_used = 0
    current_pig_alive = False        # đang có con lợn còn sống dùng lại được chưa?
    while lo < hi:
        mid = (lo + hi) // 2
        rounds += 1
        if not current_pig_alive:    # cần một con lợn mới
            pigs_used += 1
            current_pig_alive = True
        if poison <= mid:            # lợn uống [lo..mid] và chết
            current_pig_alive = False
            hi = mid
        else:                        # lợn sống, độc ở nửa phải
            lo = mid + 1
    return rounds, pigs_used


for B in [4, 1000]:
    worst_rounds = max(sequential_binary_search(B, p)[0] for p in range(B))
    worst_pigs = max(sequential_binary_search(B, p)[1] for p in range(B))
    print(f"buckets={B:>4}: tệ nhất cần {worst_rounds} lượt và {worst_pigs} con lợn")
print("Nhưng (1000, 15, 60) chỉ cho", 60 // 15, "lượt -> tìm kiếm nhị phân tuần tự KHÔNG kịp.")

buckets=   4: tệ nhất cần 2 lượt và 2 con lợn
buckets=1000: tệ nhất cần 10 lượt và 10 con lợn
Nhưng (1000, 15, 60) chỉ cho 4 lượt -> tìm kiếm nhị phân tuần tự KHÔNG kịp.


### Hai cách ngây thơ vướng ở đâu?

| Cách | Vấn đề |
|---|---|
| Mỗi con lợn thử một xô | Mỗi con lợn chỉ "nói" về **một xô** mỗi lượt ⇒ số xô tăng **tuyến tính** theo số lợn |
| Tìm kiếm nhị phân tuần tự | Mỗi lượt chỉ hỏi **một câu có/không** ⇒ tốn quá nhiều **lượt** |

**Mấu chốt cải tiến:** cho **mỗi con lợn uống rất nhiều xô cùng lúc**, sao cho **tổ hợp** "con nào chết, con nào sống" (và chết **lúc nào**) đóng vai trò như một **mã số** chỉ thẳng vào xô độc. Khi đó số xô phân biệt được tăng **lũy thừa** theo số lợn.

## 10. Cách 1 — Một lượt: mỗi con lợn là một bit

Giả sử chỉ có **1 lượt** (`T = 1`). Đánh số xô từ `0` tới `B − 1` và viết số xô dưới dạng nhị phân với `x` bit.

**Quy tắc cho uống:** con lợn `j` uống **mọi xô có bit thứ `j` bằng 1**.

Ví dụ `B = 4`, `x = 2` con lợn:

| Xô | Nhị phân (bit1 bit0) | Lợn 0 uống? (bit0) | Lợn 1 uống? (bit1) |
|:---:|:---:|:---:|:---:|
| 0 | `00` | không | không |
| 1 | `01` | **có** | không |
| 2 | `10` | không | **có** |
| 3 | `11` | **có** | **có** |

Vậy lợn 0 uống `{1, 3}`, lợn 1 uống `{2, 3}`. (Khác cách chia của đề một chút nhưng cùng tinh thần: mỗi xô một tổ hợp riêng.)

**Giải mã:** sau 15 phút, lợn `j` chết ⇔ bit thứ `j` của xô độc bằng 1. Ghép các bit lại là ra số xô.

Ví dụ: chỉ lợn 1 chết ⇒ bit1 = 1, bit0 = 0 ⇒ `10₂ = 2` ⇒ xô 2.

In [11]:
def binary_plan_one_round(buckets, pigs):
    plan = {1: [set() for _ in range(pigs)]}
    for b in range(buckets):
        bits = to_bits(b, pigs)
        for j, bit in enumerate(bits):
            if bit == 1:
                plan[1][j].add(b)
    return plan


plan8 = binary_plan_one_round(8, 3)
for j, drinks in enumerate(plan8[1]):
    print(f"Lợn {j} uống các xô {sorted(drinks)}  (các xô có bit {j} = 1)")

Lợn 0 uống các xô [1, 3, 5, 7]  (các xô có bit 0 = 1)
Lợn 1 uống các xô [2, 3, 6, 7]  (các xô có bit 1 = 1)
Lợn 2 uống các xô [4, 5, 6, 7]  (các xô có bit 2 = 1)


In [12]:
def decode_one_round(death):
    bits = [1 if t is not None else 0 for t in death]   # chết -> bit 1
    return from_digits(bits, 2)


for poison in range(8):
    death = simulate_experiment(plan8, 3, 15, 15, poison)
    guess = decode_one_round(death)
    print(f"xô độc={poison} ({format(poison, '03b')}) -> death={death} -> giải mã ra xô {guess}")
    assert guess == poison
print("Giải mã đúng cho cả 8 xô!")

xô độc=0 (000) -> death=(None, None, None) -> giải mã ra xô 0
xô độc=1 (001) -> death=(15, None, None) -> giải mã ra xô 1
xô độc=2 (010) -> death=(None, 15, None) -> giải mã ra xô 2
xô độc=3 (011) -> death=(15, 15, None) -> giải mã ra xô 3
xô độc=4 (100) -> death=(None, None, 15) -> giải mã ra xô 4
xô độc=5 (101) -> death=(15, None, 15) -> giải mã ra xô 5
xô độc=6 (110) -> death=(None, 15, 15) -> giải mã ra xô 6
xô độc=7 (111) -> death=(15, 15, 15) -> giải mã ra xô 7
Giải mã đúng cho cả 8 xô!


### Từ chiến lược tới công thức (1 lượt)

Với `x` con lợn và 1 lượt, có đúng `2^x` tổ hợp chết/sống (quy tắc nhân). Chiến lược nhị phân dùng **hết** cả `2^x` tổ hợp, nên:

- `x` con lợn phân biệt được tối đa `2^x` xô.
- Cần `x` **nhỏ nhất** sao cho `2^x ≥ buckets`.

Tìm `x` bằng vòng lặp: bắt đầu `x = 0`, còn `2^x < buckets` thì tăng `x`.

Ví dụ `buckets = 1000`: `2^9 = 512 < 1000`, `2^10 = 1024 ≥ 1000` ⇒ `x = 10`. So với `999` con lợn của cách ngây thơ!

In [13]:
def poor_pigs_one_round(buckets):
    x = 0
    while 2 ** x < buckets:
        x += 1
    return x


for B in [1, 2, 4, 5, 8, 9, 1000]:
    print(f"buckets={B:>4} -> 1 lượt cần {poor_pigs_one_round(B)} con lợn")
assert poor_pigs_one_round(4) == 2       # Example 1
assert poor_pigs_one_round(1000) == 10

buckets=   1 -> 1 lượt cần 0 con lợn
buckets=   2 -> 1 lượt cần 1 con lợn
buckets=   4 -> 1 lượt cần 2 con lợn
buckets=   5 -> 1 lượt cần 3 con lợn
buckets=   8 -> 1 lượt cần 3 con lợn
buckets=   9 -> 1 lượt cần 4 con lợn
buckets=1000 -> 1 lượt cần 10 con lợn


### Vét cạn (brute force) để chắc chắn: có thể ít lợn hơn không?

Chiến lược nhị phân *đủ dùng*, nhưng làm sao biết **không có** chiến lược thông minh hơn dùng ít lợn hơn?

Với input nhỏ ta có thể **thử hết mọi kế hoạch**. Một kế hoạch (không phụ thuộc kết quả giữa chừng) mô tả được bằng: với mỗi xô `b` và mỗi con lợn `j`, **lượt đầu tiên** mà lợn `j` uống xô `b` (`0` = không bao giờ uống). Uống lại cùng một xô ở lượt sau là vô ích: nếu xô đó độc thì lợn đã chết từ lần đầu.

Vậy mỗi xô được gán một "mã" gồm `x` số trong `{0, 1, ..., T}`. Hàm `plan_from_codes` biến các mã thành kế hoạch `plan` thật; hàm `brute_force_min_pigs` thử `x = 0, 1, 2, ...` và với mỗi `x` thử **mọi** cách gán mã, chạy simulator, xem có kế hoạch nào thành công không.

Lưu ý: hàm này viết tổng quát cho `T` lượt — ở đây ta dùng với `T = 1`, phần sau sẽ dùng lại cho `T > 1`. Nó chạy **rất chậm** (số kế hoạch là `(T+1)^(x·B)`), chỉ dùng làm "trọng tài" cho input tí hon.

In [14]:
def plan_from_codes(codes, num_pigs, T):
    # codes[b] = tuple độ dài num_pigs; codes[b][j] = lượt lợn j uống xô b (0 = không uống)
    plan = {r: [set() for _ in range(num_pigs)] for r in range(1, T + 1)}
    for b, code_b in enumerate(codes):
        for j, r in enumerate(code_b):
            if r > 0:
                plan[r][j].add(b)
    return plan


def brute_force_min_pigs(buckets, minutesToDie, minutesToTest):
    T = minutesToTest // minutesToDie
    pigs = 0
    while True:
        all_codes = list(product(range(T + 1), repeat=pigs))
        for codes in product(all_codes, repeat=buckets):
            plan = plan_from_codes(codes, pigs, T)
            ok, _ = check_plan(plan, pigs, range(buckets), minutesToDie, minutesToTest)
            if ok:
                return pigs
        pigs += 1          # không kế hoạch nào với `pigs` con thành công -> thêm 1 con


for B in range(1, 7):
    bf = brute_force_min_pigs(B, 15, 15)
    print(f"buckets={B}: vét cạn -> {bf} con | công thức 2^x >= B -> {poor_pigs_one_round(B)} con")
    assert bf == poor_pigs_one_round(B)

buckets=1: vét cạn -> 0 con | công thức 2^x >= B -> 0 con
buckets=2: vét cạn -> 1 con | công thức 2^x >= B -> 1 con
buckets=3: vét cạn -> 2 con | công thức 2^x >= B -> 2 con
buckets=4: vét cạn -> 2 con | công thức 2^x >= B -> 2 con
buckets=5: vét cạn -> 3 con | công thức 2^x >= B -> 3 con


buckets=6: vét cạn -> 3 con | công thức 2^x >= B -> 3 con


## 11. Cách 2 — Nhiều lượt: mỗi con lợn có `T + 1` trạng thái (cách chính)

Cách 1 chỉ dùng 1 lượt. Nếu có `T = 4` lượt mà vẫn dùng cách 1 thì phí 3 lượt còn lại. Câu hỏi: **thêm lượt thì mỗi con lợn "nói" được nhiều hơn bao nhiêu?**

Thử với **một con lợn** và `T` lượt:

- Lượt 1: uống xô 0. Chết ở phút `1·minutesToDie` ⇒ xô 0 độc.
- Lượt 2 (nếu còn sống): uống xô 1. Chết ở phút `2·minutesToDie` ⇒ xô 1 độc.
- …
- Lượt T: uống xô `T − 1`.
- Sống tới cuối ⇒ xô `T` (xô không ai uống) độc.

Một con lợn phân biệt được `T + 1` xô! Thông tin không chỉ nằm ở "chết hay sống" mà ở "**chết lúc nào**".

Vậy mỗi con lợn có `T + 1` **trạng thái cuối** có thể:

| Trạng thái | Ý nghĩa |
|---|---|
| 1 | chết sau lượt 1 (phút `minutesToDie`) |
| 2 | chết sau lượt 2 |
| … | … |
| T | chết sau lượt T |
| sống | không bao giờ uống phải độc |

In [15]:
def one_pig_plan(T):
    # lượt r: con lợn duy nhất uống xô r-1. Xô T không ai uống.
    return {r: [{r - 1}] for r in range(1, T + 1)}


minutesToDie, minutesToTest = 15, 60
T = minutesToTest // minutesToDie
plan1 = one_pig_plan(T)
ok, results = check_plan(plan1, 1, range(T + 1), minutesToDie, minutesToTest)
for b, res in results.items():
    print(f"xô độc = {b} -> " + ("lợn SỐNG tới cuối" if res[0] is None else f"lợn chết ở phút {res[0]}"))
print(f"1 con lợn, T={T} lượt phân biệt được {T + 1} xô? {ok}")

xô độc = 0 -> lợn chết ở phút 15
xô độc = 1 -> lợn chết ở phút 30
xô độc = 2 -> lợn chết ở phút 45
xô độc = 3 -> lợn chết ở phút 60
xô độc = 4 -> lợn SỐNG tới cuối
1 con lợn, T=4 lượt phân biệt được 5 xô? True


### Nhiều con lợn: quy tắc nhân lần nữa

Mỗi con lợn có `T + 1` trạng thái, các con lợn **độc lập** với nhau (mỗi con uống theo kế hoạch riêng). Theo quy tắc nhân, `x` con lợn có tối đa

```text
(T + 1)^x
```

tổ hợp trạng thái khác nhau. Mỗi tổ hợp có thể "đại diện" cho một xô.

Ví dụ Example 2: `T = 30 // 15 = 2` ⇒ mỗi con 3 trạng thái ⇒ 2 con lợn có `3^2 = 9` tổ hợp ≥ 4 xô. (Thực ra 2 con lợn với 30 phút xử lý được tới **9 xô**, cách chia của đề chỉ dùng 4 trong 9 tổ hợp.)

Cell dưới liệt kê đủ 9 tổ hợp cho 2 con lợn, `minutesToDie = 15`, `T = 2`.

In [16]:
pig_states = ["chết phút 15", "chết phút 30", "sống"]
all_outcomes = list(product(pig_states, repeat=2))
for i, (a, b) in enumerate(all_outcomes):
    print(f"tổ hợp {i}: lợn 0 = {a:<13} | lợn 1 = {b}")
print("Tổng:", len(all_outcomes), "= 3 ** 2")

tổ hợp 0: lợn 0 = chết phút 15  | lợn 1 = chết phút 15
tổ hợp 1: lợn 0 = chết phút 15  | lợn 1 = chết phút 30
tổ hợp 2: lợn 0 = chết phút 15  | lợn 1 = sống
tổ hợp 3: lợn 0 = chết phút 30  | lợn 1 = chết phút 15
tổ hợp 4: lợn 0 = chết phút 30  | lợn 1 = chết phút 30
tổ hợp 5: lợn 0 = chết phút 30  | lợn 1 = sống
tổ hợp 6: lợn 0 = sống          | lợn 1 = chết phút 15
tổ hợp 7: lợn 0 = sống          | lợn 1 = chết phút 30
tổ hợp 8: lợn 0 = sống          | lợn 1 = sống
Tổng: 9 = 3 ** 2


### Công thức chính và thuật toán

**Bài toán quy về:** tìm `x` nhỏ nhất sao cho

```text
(T + 1)^x >= buckets,   với T = minutesToTest // minutesToDie
```

Đặt `states = T + 1` (số trạng thái của một con lợn). Thuật toán:

1. Tính `states = minutesToTest // minutesToDie + 1`.
2. Bắt đầu `pigs = 0` (0 con lợn phân biệt được `states^0 = 1` xô — đúng trường hợp `buckets = 1`).
3. Còn `states ** pigs < buckets` thì `pigs += 1`.
4. Trả về `pigs`.

Ví dụ tính tay `buckets = 1000, 15, 60`: `T = 4`, `states = 5`.

| pigs | `5 ** pigs` | so với 1000 | Hành động |
|:---:|---:|:---:|---|
| 0 | 1 | < | tăng |
| 1 | 5 | < | tăng |
| 2 | 25 | < | tăng |
| 3 | 125 | < | tăng |
| 4 | 625 | < | tăng |
| 5 | 3125 | ≥ | **dừng, trả 5** |

In [17]:
def poor_pigs_base(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1   # T + 1 trạng thái mỗi con lợn
    pigs = 0
    while states ** pigs < buckets:
        pigs += 1
    return pigs


print(poor_pigs_base(4, 15, 15))      # Example 1 -> 2
print(poor_pigs_base(4, 15, 30))      # Example 2 -> 2
print(poor_pigs_base(1000, 15, 60))   # ví dụ kinh điển -> 5
print(poor_pigs_base(1, 1, 1))        # 1 xô -> 0

2
2
5
0


### Trace từng bước

Hàm `trace_poor_pigs` in trạng thái vòng lặp: số lợn hiện tại, số xô mà chừng đó lợn phân biệt được (`capacity`), và quyết định tăng hay dừng. Chọn 3 input:

- `(1000, 15, 60)`: ví dụ kinh điển ở bảng trên.
- `(4, 15, 15)`: trường hợp **bằng đúng** lũy thừa (`2^2 = 4`) — kiểm tra rằng dấu `<` (chứ không phải `<=`) làm ta dừng đúng lúc.
- `(1, 1, 1)`: `buckets = 1`, vòng lặp không chạy lần nào.

In [18]:
def trace_poor_pigs(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    print(f"buckets={buckets}, T={states - 1} lượt, states={states}")
    pigs = 0
    while True:
        capacity = states ** pigs
        if capacity < buckets:
            print(f"  pigs={pigs}: {states}^{pigs} = {capacity:>5} < {buckets} -> tăng")
            pigs += 1
        else:
            print(f"  pigs={pigs}: {states}^{pigs} = {capacity:>5} >= {buckets} -> dừng, trả {pigs}")
            return pigs


trace_poor_pigs(1000, 15, 60)
trace_poor_pigs(4, 15, 15)
trace_poor_pigs(1, 1, 1)

buckets=1000, T=4 lượt, states=5
  pigs=0: 5^0 =     1 < 1000 -> tăng
  pigs=1: 5^1 =     5 < 1000 -> tăng
  pigs=2: 5^2 =    25 < 1000 -> tăng
  pigs=3: 5^3 =   125 < 1000 -> tăng
  pigs=4: 5^4 =   625 < 1000 -> tăng
  pigs=5: 5^5 =  3125 >= 1000 -> dừng, trả 5
buckets=4, T=1 lượt, states=2
  pigs=0: 2^0 =     1 < 4 -> tăng
  pigs=1: 2^1 =     2 < 4 -> tăng
  pigs=2: 2^2 =     4 >= 4 -> dừng, trả 2
buckets=1, T=1 lượt, states=2
  pigs=0: 2^0 =     1 >= 1 -> dừng, trả 0


0

### Độ phức tạp của Cách 2

Gọi `x` là đáp án (số lợn). Vòng lặp chạy `x + 1` lần, mỗi lần một phép lũy thừa trên số nhỏ.

- `x ≈ log_{T+1}(buckets)`; tệ nhất `T = 1` và `buckets = 1000` cho `x = 10`.
- Thời gian: `O(x) = O(log buckets)` (có thể coi là `O(1)` với constraints của đề).
- Bộ nhớ: `O(1)`.

Có thể tối ưu nhỏ: thay vì tính lại `states ** pigs` mỗi vòng, giữ biến `capacity` và nhân dần `capacity *= states`. Ta sẽ dùng cách này trong bản LeetCode.

## 12. Cách 3 — Xây chiến lược thật và mô phỏng (chứng minh "đủ dùng")

Công thức nói "`(T+1)^x` tổ hợp", nhưng **làm sao cho lợn uống** để mỗi xô ứng với đúng một tổ hợp? Đây là cách tổng quát hóa Cách 1 từ cơ số 2 lên cơ số `T + 1`:

1. Đánh số xô `0, 1, ..., B − 1`. Viết số xô `b` trong **cơ số `T + 1`** với `x` chữ số: `d₀, d₁, ..., d_{x−1}`, mỗi chữ số thuộc `{0, 1, ..., T}`.
2. **Quy tắc cho uống:** con lợn `j` uống xô `b` ở **lượt `d_j`**. Nếu `d_j = 0` thì lợn `j` **không bao giờ uống** xô `b`.
3. **Giải mã:** lợn `j` chết ở phút `t` ⇒ `d_j = t // minutesToDie`; lợn `j` sống ⇒ `d_j = 0`. Ghép các chữ số lại bằng `from_digits` là ra xô độc.

**Vì sao lợn còn sống đúng lúc cần uống?** Gọi xô độc là `p`. Lợn `j` chỉ uống xô `p` ở lượt `d_j(p)`. Ở các lượt **trước đó**, nó chỉ uống những xô **không phải** `p` ⇒ không chết ⇒ tới lượt `d_j(p)` nó vẫn sống, uống phải độc và chết đúng sau lượt đó. Còn các lượt sau, lợn đã chết thì thôi — ta đã biết `d_j(p)` rồi.

Ví dụ `B = 9`, `T = 2` (cơ số 3), `x = 2`:

| Xô b | d₀ | d₁ | Lợn 0 uống ở lượt | Lợn 1 uống ở lượt |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | không | không |
| 1 | 1 | 0 | 1 | không |
| 2 | 2 | 0 | 2 | không |
| 3 | 0 | 1 | không | 1 |
| 4 | 1 | 1 | 1 | 1 |
| 5 | 2 | 1 | 2 | 1 |
| 6 | 0 | 2 | không | 2 |
| 7 | 1 | 2 | 1 | 2 |
| 8 | 2 | 2 | 2 | 2 |

In [19]:
def build_feeding_plan(buckets, pigs, T):
    base = T + 1
    codes = [tuple(to_digits(b, base, pigs)) for b in range(buckets)]
    return plan_from_codes(codes, pigs, T)     # tái dùng hàm ở phần vét cạn


plan9 = build_feeding_plan(9, 2, 2)
for r in sorted(plan9):
    for j in range(2):
        print(f"Lượt {r}: lợn {j} uống các xô {sorted(plan9[r][j])}")
print()


def decode_death_times(death, minutesToDie, base):
    digits = [0 if t is None else t // minutesToDie for t in death]
    return from_digits(digits, base)


for poison in range(9):
    death = simulate_experiment(plan9, 2, 15, 30, poison)
    print(f"xô độc={poison} -> death={death} -> giải mã {decode_death_times(death, 15, 3)}")

Lượt 1: lợn 0 uống các xô [1, 4, 7]
Lượt 1: lợn 1 uống các xô [3, 4, 5]
Lượt 2: lợn 0 uống các xô [2, 5, 8]
Lượt 2: lợn 1 uống các xô [6, 7, 8]

xô độc=0 -> death=(None, None) -> giải mã 0
xô độc=1 -> death=(15, None) -> giải mã 1
xô độc=2 -> death=(30, None) -> giải mã 2
xô độc=3 -> death=(None, 15) -> giải mã 3
xô độc=4 -> death=(15, 15) -> giải mã 4
xô độc=5 -> death=(30, 15) -> giải mã 5
xô độc=6 -> death=(None, 30) -> giải mã 6
xô độc=7 -> death=(15, 30) -> giải mã 7
xô độc=8 -> death=(30, 30) -> giải mã 8


### Trace một thí nghiệm thật: 9 xô, `minutesToDie = 15`, `minutesToTest = 30`, xô độc = 7

Tính tay trước: `7 = 1 + 2·3` ⇒ `d₀ = 1`, `d₁ = 2`.

| Phút | Sự kiện | Lợn 0 | Lợn 1 |
|---:|---|---|---|
| 0 | Lượt 1: lợn 0 uống `{1,4,7}`, lợn 1 uống `{3,4,5}` | uống phải 7 | an toàn |
| 15 | Xem kết quả lượt 1 | **chết** ⇒ `d₀ = 15 // 15 = 1` | sống |
| 15 | Lượt 2: lợn 0 đã chết; lợn 1 uống `{6,7,8}` | — | uống phải 7 |
| 30 | Xem kết quả lượt 2 | — | **chết** ⇒ `d₁ = 30 // 15 = 2` |
| 30 | Giải mã | | `1 + 2·3 = 7` ✓ |

Hàm `trace_experiment` in lại đúng diễn biến này.

In [20]:
def trace_experiment(buckets, minutesToDie, minutesToTest, poison):
    T = minutesToTest // minutesToDie
    pigs = poor_pigs_base(buckets, minutesToDie, minutesToTest)
    base = T + 1
    plan = build_feeding_plan(buckets, pigs, T)
    print(f"{buckets} xô, T={T}, cơ số {base}, {pigs} con lợn; xô độc = {poison},"
          f" mã = {to_digits(poison, base, pigs)}")
    alive = [True] * pigs
    death = [None] * pigs
    for r in range(1, T + 1):
        for j in range(pigs):
            if not alive[j]:
                print(f"  phút {(r - 1) * minutesToDie:>3}: lợn {j} đã chết, bỏ qua")
                continue
            drinks = sorted(plan[r][j])
            hit = poison in plan[r][j]
            print(f"  phút {(r - 1) * minutesToDie:>3}: lợn {j} uống {drinks}"
                  f" -> {'UỐNG PHẢI ĐỘC' if hit else 'an toàn'}")
            if hit:
                alive[j] = False
                death[j] = r * minutesToDie
        print(f"  phút {r * minutesToDie:>3}: xem kết quả lượt {r}, death = {death}")
    found = decode_death_times(death, minutesToDie, base)
    print(f"  Giải mã: chữ số {[0 if t is None else t // minutesToDie for t in death]} -> xô {found}")
    return found


assert trace_experiment(9, 15, 30, poison=7) == 7

9 xô, T=2, cơ số 3, 2 con lợn; xô độc = 7, mã = [1, 2]
  phút   0: lợn 0 uống [1, 4, 7] -> UỐNG PHẢI ĐỘC
  phút   0: lợn 1 uống [3, 4, 5] -> an toàn
  phút  15: xem kết quả lượt 1, death = [15, None]
  phút  15: lợn 0 đã chết, bỏ qua
  phút  15: lợn 1 uống [6, 7, 8] -> UỐNG PHẢI ĐỘC
  phút  30: xem kết quả lượt 2, death = [15, 30]
  Giải mã: chữ số [1, 2] -> xô 7


### Kiểm chứng cho MỌI xô, nhiều bộ tham số

Bây giờ ta chạy chiến lược cơ số `T + 1` với **đúng `x = poor_pigs_base(...)` con lợn** cho nhiều bộ tham số, và với **từng xô** làm xô độc:

- Simulator phải không báo lỗi "quá số lượt" (tức chiến lược kết thúc trong `minutesToTest`).
- Mọi xô cho kết quả khác nhau, và giải mã ra đúng xô.

Có cả `(1000, 15, 60)`: 5 con lợn, 4 lượt, 1000 xô — kiểm tra toàn bộ 1000 trường hợp.

In [21]:
def verify_strategy(buckets, minutesToDie, minutesToTest):
    T = minutesToTest // minutesToDie
    pigs = poor_pigs_base(buckets, minutesToDie, minutesToTest)
    plan = build_feeding_plan(buckets, pigs, T)
    for poison in range(buckets):
        death = simulate_experiment(plan, pigs, minutesToDie, minutesToTest, poison)
        if decode_death_times(death, minutesToDie, T + 1) != poison:
            return pigs, False
    return pigs, True


configs = [(4, 15, 15), (4, 15, 30), (9, 15, 30), (1000, 15, 60),
           (125, 15, 60), (1000, 1, 100), (1000, 100, 100), (1, 1, 1), (2, 10, 10)]
for cfg in configs:
    pigs, ok = verify_strategy(*cfg)
    print(f"{str(cfg):<16} -> {pigs:>2} con lợn, giải mã đúng mọi xô: {ok}")
    assert ok

(4, 15, 15)      ->  2 con lợn, giải mã đúng mọi xô: True
(4, 15, 30)      ->  2 con lợn, giải mã đúng mọi xô: True
(9, 15, 30)      ->  2 con lợn, giải mã đúng mọi xô: True
(1000, 15, 60)   ->  5 con lợn, giải mã đúng mọi xô: True
(125, 15, 60)    ->  3 con lợn, giải mã đúng mọi xô: True


(1000, 1, 100)   ->  2 con lợn, giải mã đúng mọi xô: True
(1000, 100, 100) -> 10 con lợn, giải mã đúng mọi xô: True


(1, 1, 1)        ->  0 con lợn, giải mã đúng mọi xô: True
(2, 10, 10)      ->  1 con lợn, giải mã đúng mọi xô: True


### Chiều ngược lại: vì sao KHÔNG thể ít lợn hơn?

Cách 3 chứng minh `x` con lợn là **đủ**. Còn phải chứng minh `x − 1` con là **không đủ**, với **mọi** chiến lược (kể cả chiến lược quyết định cho uống tùy theo kết quả các lượt trước — adaptive):

1. Kết thúc thí nghiệm, điều duy nhất ta quan sát được là **mỗi con lợn chết sau lượt nào, hay sống**. Mỗi con có tối đa `T + 1` khả năng.
2. Vậy với `x − 1` con, có tối đa `(T + 1)^(x − 1)` kết quả khác nhau.
3. Vì `x` là số **nhỏ nhất** thỏa `(T+1)^x ≥ B`, ta có `(T + 1)^(x − 1) < B`.
4. Theo **nguyên lý chuồng bồ câu**, có hai xô khác nhau cho **cùng** một kết quả ⇒ không phân biệt được ⇒ thất bại.

Chiến lược adaptive cũng không thoát được: dù quyết định khéo thế nào, mỗi xô độc vẫn dẫn tới **một** kết quả trong `(T+1)^(x−1)` kết quả có thể.

Dưới đây ta chạy "trọng tài vét cạn" `brute_force_min_pigs` (thử mọi kế hoạch cố định) với `T > 1` trên input tí hon, và so với công thức.

In [22]:
for B in range(1, 7):
    for die, test in [(15, 30), (15, 45)]:
        bf = brute_force_min_pigs(B, die, test)
        formula = poor_pigs_base(B, die, test)
        assert bf == formula, (B, die, test, bf, formula)
    print(f"buckets={B}: T=2 -> {poor_pigs_base(B, 15, 30)} con, T=3 -> {poor_pigs_base(B, 15, 45)} con"
          f"  (vét cạn khớp công thức)")

buckets=1: T=2 -> 0 con, T=3 -> 0 con  (vét cạn khớp công thức)
buckets=2: T=2 -> 1 con, T=3 -> 1 con  (vét cạn khớp công thức)
buckets=3: T=2 -> 1 con, T=3 -> 1 con  (vét cạn khớp công thức)
buckets=4: T=2 -> 2 con, T=3 -> 1 con  (vét cạn khớp công thức)


buckets=5: T=2 -> 2 con, T=3 -> 2 con  (vét cạn khớp công thức)


buckets=6: T=2 -> 2 con, T=3 -> 2 con  (vét cạn khớp công thức)


## 13. Cách 4 — Công thức "một dòng" bằng logarit và cái bẫy sai số

Từ `(T+1)^x ≥ B`, lấy logarit hai vế: `x ≥ log_{T+1}(B) = ln(B) / ln(T+1)`. Vì `x` nguyên nhỏ nhất:

```python
x = math.ceil(math.log(buckets) / math.log(states))
```

Rất nhiều lời giải trên mạng viết đúng một dòng này. Nhưng **số thực (float) trên máy tính có sai số làm tròn**: máy lưu số ở dạng nhị phân với số bit hữu hạn, nhiều số như `0.1` hay `log(5)` **không biểu diễn chính xác** được.

Hậu quả: khi `buckets` là **lũy thừa đúng** của `states`, kết quả logarit đáng lẽ là số nguyên lại lệch đi một chút:

- Lệch **lên** (`3.0000000000000004`) ⇒ `ceil` cho `4` thay vì `3` ⇒ **sai**.
- Lệch **xuống** (`2.9999999999999996`) ⇒ `ceil` vẫn đúng, nhưng ai dùng `int(...)` hay `floor` thì sai.

Hãy xem tận mắt.

In [23]:
import math

print("0.1 + 0.2 =", 0.1 + 0.2, "| bằng 0.3?", 0.1 + 0.2 == 0.3)
print()
print("log(125)/log(5)  =", math.log(125) / math.log(5))
print("log(216)/log(6)  =", math.log(216) / math.log(6))
print("log(1000, 10)    =", math.log(1000, 10))
print()
# buckets = 125, minutesToDie = 15, minutesToTest = 60  -> states = 5, đáp án đúng 3 (5^3 = 125)
x_log = math.ceil(math.log(125) / math.log(5))
print("Cách log cho (125, 15, 60):", x_log, "| đúng là:", poor_pigs_base(125, 15, 60))

0.1 + 0.2 = 0.30000000000000004 | bằng 0.3? False

log(125)/log(5)  = 3.0000000000000004
log(216)/log(6)  = 3.0000000000000004
log(1000, 10)    = 2.9999999999999996

Cách log cho (125, 15, 60): 4 | đúng là: 3


### Quét toàn bộ miền input để thấy quy mô lỗi

Constraints nhỏ nên ta có thể thử **mọi** cặp `(buckets, states)` với `buckets ∈ [1, 1000]`, `states ∈ [2, 101]` (100 000 cặp) và đếm bao nhiêu chỗ công thức log sai.

Sau đó viết bản **sửa lỗi**: dùng log để ước lượng nhanh, rồi **chỉnh lại bằng số nguyên** (hai vòng `while` nhỏ đẩy `x` về đúng vị trí). Cách an toàn nhất vẫn là vòng lặp số nguyên thuần túy của Cách 2 — đơn giản, không cần nghĩ về float.

In [24]:
def poor_pigs_log_naive(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    return math.ceil(math.log(buckets) / math.log(states))


def poor_pigs_log_fixed(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    x = math.ceil(math.log(buckets) / math.log(states))   # ước lượng (có thể lệch 1)
    while x > 0 and states ** (x - 1) >= buckets:         # lệch lên -> hạ xuống
        x -= 1
    while states ** x < buckets:                          # lệch xuống -> nâng lên
        x += 1
    return x


wrong = []
for states in range(2, 102):
    die, test = 1, states - 1            # tạo minutesToDie/minutesToTest cho đúng states
    for B in range(1, 1001):
        good = poor_pigs_base(B, die, test)
        assert poor_pigs_log_fixed(B, die, test) == good
        if poor_pigs_log_naive(B, die, test) != good:
            wrong.append((B, states, poor_pigs_log_naive(B, die, test), good))
print("Số cặp (buckets, states) mà log naive SAI:", len(wrong))
for B, states, got, good in wrong:
    print(f"  buckets={B}, states={states}: log naive = {got}, đúng = {good}")
print("poor_pigs_log_fixed đúng trên cả 100 000 cặp: PASS")

Số cặp (buckets, states) mà log naive SAI: 2
  buckets=125, states=5: log naive = 4, đúng = 3
  buckets=216, states=6: log naive = 4, đúng = 3
poor_pigs_log_fixed đúng trên cả 100 000 cặp: PASS


**Bài học:** chỉ 2 trên 100 000 trường hợp sai — đủ hiếm để bạn tự test vài ví dụ không phát hiện ra, nhưng đủ để một bộ test kỹ bắt lỗi. Quy tắc chung:

> Khi bài toán **thuần số nguyên** (đếm, so sánh lũy thừa), hãy giải bằng **phép toán số nguyên**. Chỉ dùng float khi thật cần, và nếu dùng thì phải **chỉnh lại bằng số nguyên**.

Một số người thêm "epsilon" như `math.ceil(math.log(B) / math.log(s) - 1e-9)`. Cách này chạy được ở đây nhưng là "chắp vá": phải tự chọn epsilon và tự tin rằng nó đủ lớn mà không quá lớn. Vòng lặp số nguyên thì **không có gì phải lo**.

## 14. Code nộp LeetCode

Bản nộp dùng **Cách 2** với vòng lặp số nguyên. Thay vì tính `states ** pigs` mỗi vòng, giữ biến `capacity` = số xô mà `pigs` con lợn phân biệt được, mỗi lần thêm một con lợn thì `capacity *= states`.

- `states = minutesToTest // minutesToDie + 1`: chia **lấy phần nguyên** và **cộng 1** cho trạng thái "sống".
- `pigs = 0, capacity = 1`: không con lợn nào vẫn "phân biệt" được 1 xô.
- `while capacity < buckets`: dấu `<`, không phải `<=`.

In [25]:
class Solution:
    def poorPigs(self, buckets: int, minutesToDie: int, minutesToTest: int) -> int:
        states = minutesToTest // minutesToDie + 1
        pigs = 0
        capacity = 1
        while capacity < buckets:
            capacity *= states
            pigs += 1
        return pigs


sol = Solution()
tests = [
    ((4, 15, 15), 2),       # Example 1
    ((4, 15, 30), 2),       # Example 2
    ((1000, 15, 60), 5),    # ví dụ kinh điển
    ((1, 1, 1), 0),         # 1 xô: không cần lợn
    ((2, 1, 1), 1),         # 2 xô, 1 lượt
    ((125, 15, 60), 3),     # đúng bằng 5^3 -- bẫy của cách log
    ((126, 15, 60), 4),     # vượt 5^3 một chút
    ((1000, 1, 1), 10),     # 1 lượt: 2^10 = 1024
    ((1000, 1, 100), 2),    # 101 trạng thái: 101^2 = 10201
    ((1000, 15, 40), 7),    # T = 2 (10 phút dư vô dụng): 3^6=729 < 1000 <= 3^7
]
for args, expected in tests:
    got = sol.poorPigs(*args)
    print(f"poorPigs{args} = {got} (expected {expected})")
    assert got == expected
print("Tất cả test PASS!")

poorPigs(4, 15, 15) = 2 (expected 2)
poorPigs(4, 15, 30) = 2 (expected 2)
poorPigs(1000, 15, 60) = 5 (expected 5)
poorPigs(1, 1, 1) = 0 (expected 0)
poorPigs(2, 1, 1) = 1 (expected 1)
poorPigs(125, 15, 60) = 3 (expected 3)
poorPigs(126, 15, 60) = 4 (expected 4)
poorPigs(1000, 1, 1) = 10 (expected 10)
poorPigs(1000, 1, 100) = 2 (expected 2)
poorPigs(1000, 15, 40) = 7 (expected 7)
Tất cả test PASS!


### Test đối chiếu chéo các cách trên input ngẫu nhiên

Ta so sánh trên input ngẫu nhiên (cố định `random.seed` để chạy lại ra y hệt):

- `Solution().poorPigs`, `poor_pigs_base`, `poor_pigs_log_fixed` phải trả cùng một số.
- Khi `T = 1`, kết quả phải bằng `poor_pigs_one_round` (Cách 1).
- Chiến lược thật (Cách 3) với đúng số lợn đó phải giải mã đúng **mọi** xô (`verify_strategy`) — chạy trên một phần các input vì mô phỏng tốn thời gian hơn.
- Với `x − 1` con lợn, số kết quả `(T+1)^(x−1)` phải **nhỏ hơn** `buckets` (chiều "không thể ít hơn").
- Input tí hon (`buckets ≤ 5`) thì đối chiếu thêm với trọng tài vét cạn.

In [26]:
random.seed(458)
count = 0
for i in range(400):
    B = random.randint(1, 1000)
    die = random.randint(1, 100)
    test = random.randint(die, 100)
    T = test // die
    x = Solution().poorPigs(B, die, test)
    assert x == poor_pigs_base(B, die, test) == poor_pigs_log_fixed(B, die, test)
    if T == 1:
        assert x == poor_pigs_one_round(B)
    assert (T + 1) ** x >= B
    assert x == 0 or (T + 1) ** (x - 1) < B
    if i < 25:
        assert verify_strategy(B, die, test)[1]
    count += 1

for B in range(1, 6):
    for die, test in [(10, 10), (10, 20), (10, 35)]:
        assert brute_force_min_pigs(B, die, test) == Solution().poorPigs(B, die, test)
        count += 1
print(f"{count} bộ test đối chiếu chéo: PASS")

415 bộ test đối chiếu chéo: PASS


## 15. Vì sao lời giải đúng? (tóm tắt chứng minh)

Gọi `T = minutesToTest // minutesToDie`, `s = T + 1`, `x` = số nhỏ nhất với `s^x ≥ buckets`.

**(a) `x` con lợn là đủ — chứng minh bằng cách xây dựng (Cách 3).** Viết số xô trong cơ số `s` với `x` chữ số (đủ chỗ vì `s^x ≥ buckets`). Lợn `j` uống xô `b` ở lượt `d_j(b)` (0 = không uống). Với xô độc `p`, lợn `j` sống qua các lượt trước `d_j(p)` (chỉ uống xô sạch), chết đúng sau lượt `d_j(p)`, hoặc sống mãi nếu `d_j(p) = 0`. Đọc thời điểm chết ⇒ đọc ra đúng mọi chữ số của `p` ⇒ biết `p`. Tổng thời gian `T · minutesToDie ≤ minutesToTest`.

**(b) `x − 1` con lợn là không đủ — chứng minh bằng đếm.** Mỗi con lợn chỉ có `s` kết cục quan sát được, nên `x − 1` con có tối đa `s^(x−1) < buckets` kết quả. Theo nguyên lý chuồng bồ câu, hai xô nào đó cho cùng kết quả ⇒ không phân biệt được. Lập luận này đúng cho **mọi** chiến lược, kể cả adaptive.

**Bất biến (invariant) của vòng lặp** trong `Solution`: ở đầu mỗi vòng, `capacity = s^pigs` và mọi số lợn nhỏ hơn `pigs` đều **không đủ** (`s^k < buckets` với mọi `k < pigs`). Vòng lặp dừng ở lần đầu `capacity ≥ buckets` ⇒ `pigs` chính là `x` nhỏ nhất.

Một nhận xét thú vị: số lợn **không phụ thuộc** vào việc chiến lược có "thông minh" hay không — giới hạn đến từ **lượng thông tin** quan sát được. Đây là kiểu lập luận "**chặn dưới thông tin**" (information-theoretic lower bound), giống như lý do tìm kiếm trên `n` phần tử bằng so sánh cần ít nhất `log₂ n` câu hỏi.

## 16. So sánh các cách

Gọi `B = buckets`, `T` = số lượt, `x` = đáp án.

| Cách | Ý tưởng | Số lợn cần (B=1000, 15, 60) | Thời gian tính | Ghi chú |
|---|---|:---:|---|---|
| Ngây thơ A | Mỗi con lợn thử 1 xô / lượt | `ceil(999/4) = 250` | `O(1)` | Đúng nhưng tốn lợn khủng khiếp, tăng tuyến tính |
| Ngây thơ B | Tìm kiếm nhị phân tuần tự | cần 10 lượt > 4 ⇒ **không kịp** | — | Chỉ hỏi 1 câu có/không mỗi lượt |
| Cách 1 | 1 lượt, mỗi lợn = 1 bit, `2^x ≥ B` | `10` (bỏ phí 3 lượt) | `O(log B)` | Đúng khi `T = 1`, nền tảng của cách chính |
| **Cách 2** | Mỗi lợn `T+1` trạng thái, `(T+1)^x ≥ B` | **`5`** | `O(log B)`, bộ nhớ `O(1)` | **Lời giải chính, nộp LeetCode** |
| Cách 3 | Mã cơ số `T+1` + mô phỏng + giải mã | `5` | `O(T · x · B)` để mô phỏng mọi xô | Không cần để nộp bài; dùng để **chứng minh** và kiểm chứng |
| Cách 4 | `ceil(log B / log(T+1))` | `5` (nhưng sai ở `B=125`!) | `O(1)` | Bẫy float; phải chỉnh lại bằng số nguyên |

## 17. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `/` thay vì `//`

```python
states = minutesToTest / minutesToDie + 1     # SAI: 40 / 15 + 1 = 3.666...
```

Lượt thử không thể "chia lẻ": 40 phút với `minutesToDie = 15` chỉ có 2 lượt. Với `(10, 15, 40)`: đúng là `3^2 = 9 < 10` ⇒ cần 3 con; bản sai thấy `3.67^2 ≈ 13.4 ≥ 10` ⇒ trả 2.

### Lỗi 2: Quên `+ 1` (quên trạng thái "sống")

```python
states = minutesToTest // minutesToDie        # SAI: thiếu trạng thái "sống"
```

Khi `T = 1` thì `states = 1`, `1 ** pigs` mãi bằng 1 ⇒ **vòng lặp vô hạn**! Khi `T = 2`: `(9, 15, 30)` trả 4 thay vì 2.

### Lỗi 3: Dùng `<=` thay vì `<`

```python
while states ** pigs <= buckets:   # SAI khi buckets là lũy thừa đúng
```

`(4, 15, 15)`: `2^2 = 4 <= 4` vẫn tăng ⇒ trả 3 thay vì 2.

### Lỗi 4: Dùng `math.log` không chỉnh lại

`(125, 15, 60)` trả 4 thay vì 3 (đã thấy ở Cách 4).

### Lỗi 5: Bắt đầu `pigs = 1`, hoặc trả "ít nhất 1 con"

`buckets = 1` thì đáp án là `0`. Bắt đầu đếm từ 1 sẽ trả 1.

### Lỗi 6: Nghĩ rằng mỗi con lợn chỉ uống một xô

Đây là lỗi **tư duy** chứ không phải lỗi code: đề cho phép một con lợn uống **nhiều xô cùng lúc**, và chính điều này tạo ra mã hóa lũy thừa.

Cell dưới chạy các phiên bản sai (có giới hạn số vòng để không treo máy) và so với bản đúng.

In [27]:
def wrong_float_division(buckets, minutesToDie, minutesToTest):
    states = minutesToTest / minutesToDie + 1          # lỗi 1
    pigs = 0
    while states ** pigs < buckets:
        pigs += 1
    return pigs


def wrong_no_plus_one(buckets, minutesToDie, minutesToTest, max_iter=50):
    states = minutesToTest // minutesToDie             # lỗi 2
    pigs = 0
    while states ** pigs < buckets:
        pigs += 1
        if pigs > max_iter:
            return "VÒNG LẶP VÔ HẠN"
    return pigs


def wrong_less_equal(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    pigs = 0
    while states ** pigs <= buckets:                   # lỗi 3
        pigs += 1
    return pigs


def wrong_start_one(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    pigs = 1                                           # lỗi 5
    while states ** pigs < buckets:
        pigs += 1
    return pigs


cases = [
    ("lỗi 1: dùng /", wrong_float_division, (10, 15, 40)),
    ("lỗi 2: thiếu +1", wrong_no_plus_one, (9, 15, 30)),
    ("lỗi 2: thiếu +1", wrong_no_plus_one, (4, 15, 15)),
    ("lỗi 3: dùng <=", wrong_less_equal, (4, 15, 15)),
    ("lỗi 4: log naive", poor_pigs_log_naive, (125, 15, 60)),
    ("lỗi 5: pigs = 1", wrong_start_one, (1, 1, 1)),
]
for name, fn, args in cases:
    print(f"{name:<17} {str(args):<14} -> sai: {str(fn(*args)):<16} | đúng: {Solution().poorPigs(*args)}")

lỗi 1: dùng /     (10, 15, 40)   -> sai: 2                | đúng: 3
lỗi 2: thiếu +1   (9, 15, 30)    -> sai: 4                | đúng: 2
lỗi 2: thiếu +1   (4, 15, 15)    -> sai: VÒNG LẶP VÔ HẠN  | đúng: 2
lỗi 3: dùng <=    (4, 15, 15)    -> sai: 3                | đúng: 2
lỗi 4: log naive  (125, 15, 60)  -> sai: 4                | đúng: 3
lỗi 5: pigs = 1   (1, 1, 1)      -> sai: 1                | đúng: 0


## 18. Mẹo nhận dạng pattern

Dấu hiệu bài thuộc dạng "**mã hóa thông tin / đếm kết quả**":

- Đề hỏi **số lượng tối thiểu** (lợn, lần cân, câu hỏi, thí nghiệm…) để **phân biệt** một trong `N` khả năng.
- Mỗi "đơn vị" (con lợn, lần cân) cho ra một **số hữu hạn kết quả quan sát được**.

Công thức tư duy:

```text
1. Mỗi đơn vị cho ra bao nhiêu kết quả? -> k
2. x đơn vị độc lập -> k^x tổ hợp (quy tắc nhân)
3. Cần k^x >= N  -> x nhỏ nhất (vòng lặp số nguyên)
4. Chiều "đủ": xây mã cơ số k cho từng khả năng
5. Chiều "cần": nguyên lý chuồng bồ câu
```

Câu hỏi then chốt cần tự hỏi: "**Một con lợn có thể cho mình biết bao nhiêu điều?**" — câu trả lời không phải 2 (chết/sống) mà là `T + 1` (chết lúc nào).

## 19. Biến thể và bài liên quan

- **Biến thể "ngược"**: cho số lợn `p` và thời gian, hỏi tối đa bao nhiêu xô? ⇒ `(T + 1)^p`.
- **Biến thể "thời gian"**: cho `buckets` và số lợn `p`, hỏi cần `minutesToTest` ít nhất bao nhiêu? ⇒ tìm `T` nhỏ nhất với `(T + 1)^p ≥ buckets`, trả `T · minutesToDie`.
- **Bài toán cân đồng xu**: `N` đồng xu, một đồng nặng hơn, cân đĩa mỗi lần có 3 kết quả (trái nặng / phải nặng / cân bằng) ⇒ số lần cân ít nhất là `w` nhỏ nhất với `3^w ≥ N`. Cùng một khuôn tư duy!
- **Bài 20 câu hỏi có/không**: đoán số trong `1..N` cần `ceil(log₂ N)` câu — tương đương Cách 1.
- Bài liên quan trên LeetCode: **#887 Super Egg Drop**, **#1884 Egg Drop With 2 Eggs and N Floors** (cũng "đếm số khả năng phân biệt được" nhưng thí nghiệm có thứ tự, phải dùng DP), **#374 Guess Number Higher or Lower** (binary search, chặn dưới `log₂ n`).

Cell dưới minh họa nhanh ba biến thể đầu bằng đúng cùng một khuôn vòng lặp số nguyên.

In [28]:
def max_buckets(pigs, minutesToDie, minutesToTest):
    return (minutesToTest // minutesToDie + 1) ** pigs


def min_weighings(n_coins):
    w, capacity = 0, 1
    while capacity < n_coins:
        capacity *= 3
        w += 1
    return w


print("5 con lợn, (15, 60) xử lý tối đa", max_buckets(5, 15, 60), "xô")
print("2 con lợn, (15, 30) xử lý tối đa", max_buckets(2, 15, 30), "xô")
for n in [3, 9, 10, 27, 1000]:
    print(f"{n:>4} đồng xu (1 đồng nặng hơn) -> cần {min_weighings(n)} lần cân")

5 con lợn, (15, 60) xử lý tối đa 3125 xô
2 con lợn, (15, 30) xử lý tối đa 9 xô
   3 đồng xu (1 đồng nặng hơn) -> cần 1 lần cân
   9 đồng xu (1 đồng nặng hơn) -> cần 2 lần cân
  10 đồng xu (1 đồng nặng hơn) -> cần 3 lần cân
  27 đồng xu (1 đồng nặng hơn) -> cần 3 lần cân
1000 đồng xu (1 đồng nặng hơn) -> cần 7 lần cân


## 20. Bài tập vận dụng

Làm theo thứ tự từ dễ tới khó. **Đáp án ở cuối notebook** — hãy tự làm trước.

### Bài 1 — Tính tay (không chạy code)

Tính đáp án `poorPigs` cho các input sau, ghi rõ `T`, `states` và lũy thừa dùng để so sánh:

1. `buckets = 27, minutesToDie = 10, minutesToTest = 30`
2. `buckets = 1000, minutesToDie = 15, minutesToTest = 15`
3. `buckets = 25, minutesToDie = 12, minutesToTest = 59`
4. `buckets = 1, minutesToDie = 5, minutesToTest = 100`

Thêm câu hỏi: với `9` xô, `minutesToDie = 15`, `minutesToTest = 30` và chiến lược Cách 3, nếu lợn 0 **sống** và lợn 1 **chết ở phút 15** thì xô nào độc?

Hãy viết ra giấy rồi so với phần đáp án cuối notebook.

### Bài 2 — Điền TODO cho lời giải chính

Hoàn thiện hàm theo đúng 3 bước: tính `states`, khởi tạo, vòng lặp nhân dần `capacity`.

In [29]:
def poor_pigs_exercise(buckets, minutesToDie, minutesToTest):
    # TODO 1: states = số trạng thái của một con lợn (nhớ // và + 1)
    # TODO 2: pigs = 0, capacity = 1
    # TODO 3: còn capacity < buckets thì nhân capacity với states và tăng pigs
    # TODO 4: trả về pigs
    pass

# Tự làm xong rồi mới mở test:
# assert poor_pigs_exercise(4, 15, 15) == 2
# assert poor_pigs_exercise(4, 15, 30) == 2
# assert poor_pigs_exercise(1000, 15, 60) == 5
# assert poor_pigs_exercise(125, 15, 60) == 3
# assert poor_pigs_exercise(1, 1, 1) == 0

### Bài 3 — Tự viết bộ giải mã

Viết hàm `find_poison(death_minutes, minutesToDie, minutesToTest)`:

- `death_minutes[j]` là phút lợn `j` chết, hoặc `None` nếu sống.
- Chiến lược đã dùng là Cách 3 (lợn `j` uống xô `b` ở lượt bằng chữ số thứ `j` của `b` trong cơ số `T + 1`).
- Trả về số xô độc (đánh số từ 0).

**Không** dùng lại `decode_death_times` hay `from_digits` — tự viết từ đầu.

Ví dụ: `find_poison([15, 30], 15, 30) == 7` (vì `1 + 2·3 = 7`).

In [30]:
def find_poison(death_minutes, minutesToDie, minutesToTest):
    # TODO: tính base = T + 1
    # TODO: với mỗi con lợn j, đổi phút chết thành chữ số (None -> 0)
    # TODO: cộng dồn chữ số * base**j
    raise NotImplementedError

# Tự làm xong rồi mới mở test:
# assert find_poison([15, 30], 15, 30) == 7
# assert find_poison([None, None], 15, 30) == 0
# assert find_poison([None, 15], 15, 30) == 3
# assert find_poison([45, None, 15, 60, 30], 15, 60) == 3 + 1 * 25 + 4 * 125 + 2 * 625

### Bài 4 — Biến thể: thời gian tối thiểu

Cho `buckets`, số lợn `pigs` và `minutesToDie`. Hỏi cần `minutesToTest` **ít nhất** là bao nhiêu phút để chắc chắn tìm ra xô độc? Nếu không thể (ví dụ `pigs = 0` mà `buckets > 1`) trả `-1`.

Gợi ý đầu bài: tìm số lượt `T` nhỏ nhất sao cho `(T + 1)^pigs ≥ buckets`.

Ví dụ: `min_minutes_to_test(1000, 5, 15) == 45` (vì `4^5 = 1024 ≥ 1000` nên chỉ cần 3 lượt), `min_minutes_to_test(4, 2, 15) == 15`, `min_minutes_to_test(1, 0, 15) == 0`.

In [31]:
def min_minutes_to_test(buckets, pigs, minutesToDie):
    # TODO 1: xử lý buckets == 1 (không cần thời gian)
    # TODO 2: xử lý pigs == 0 (không thể nếu buckets > 1)
    # TODO 3: tăng T từ 0 cho tới khi (T + 1) ** pigs >= buckets
    # TODO 4: trả về T * minutesToDie
    pass

# Tự làm xong rồi mới mở test:
# assert min_minutes_to_test(1000, 5, 15) == 45
# assert min_minutes_to_test(4, 2, 15) == 15
# assert min_minutes_to_test(9, 2, 15) == 30
# assert min_minutes_to_test(10, 2, 15) == 45
# assert min_minutes_to_test(1, 0, 15) == 0
# assert min_minutes_to_test(2, 0, 15) == -1

### Bài 5 — Chuyển giao kỹ thuật: cân đồng xu (có ghi lại chiến lược)

Có `n` đồng xu, đúng một đồng **nặng hơn**. Mỗi lần cân đĩa cho 3 kết quả: đĩa trái nặng hơn, đĩa phải nặng hơn, hoặc cân bằng.

Viết `coin_strategy(n)` trả về `(w, plan)`:

- `w` = số lần cân ít nhất (nhỏ nhất với `3^w ≥ n`).
- `plan` là list độ dài `w`; `plan[i] = (left, right)` là **hai tập** đồng xu đặt lên đĩa trái / phải ở lần cân thứ `i`. Dùng mã cơ số 3: đồng `c` có chữ số `i` bằng `1` thì đặt trái, bằng `2` thì đặt phải, bằng `0` thì để ngoài.

Thêm hàm `weigh_and_decode(n, plan, heavy)` mô phỏng các lần cân với đồng nặng là `heavy` và giải mã ra đồng nặng.

*Lưu ý (giả định đơn giản hóa):* trong bài cân thật, hai đĩa phải có **cùng số đồng** thì kết quả mới đáng tin, và cần chiến lược khéo hơn để giữ cân bằng số lượng. Bài này chỉ luyện phần **mã hóa / giải mã**, nên ta giả sử cân "báo" cho biết đồng nặng nằm ở đĩa trái, đĩa phải hay ở ngoài — giống như con lợn "báo" nó đã uống phải độc ở lượt nào.

In [32]:
def coin_strategy(n):
    # TODO 1: tìm w nhỏ nhất với 3 ** w >= n
    # TODO 2: với mỗi lần cân i, tạo left = các đồng có chữ số i bằng 1,
    #         right = các đồng có chữ số i bằng 2 (dùng to_digits(c, 3, w))
    pass


def weigh_and_decode(n, plan, heavy):
    # TODO 1: với mỗi lần cân i: heavy in left -> chữ số 1; heavy in right -> 2; còn lại -> 0
    # TODO 2: ghép chữ số cơ số 3 thành số đồng xu
    pass

# Tự làm xong rồi mới mở test:
# w, plan = coin_strategy(27)
# assert w == 3
# assert all(weigh_and_decode(27, plan, c) == c for c in range(27))
# assert coin_strategy(28)[0] == 4 and coin_strategy(1)[0] == 0

### Bài 6 — Mini challenge: tự xây chiến lược và kiểm chứng bằng simulator

Viết `my_feeding_plan(buckets, minutesToDie, minutesToTest)` trả về `(pigs, plan)` với `plan` đúng định dạng của `simulate_experiment` (`plan[r][j]` = tập xô lợn `j` uống ở lượt `r`), **không** dùng lại `build_feeding_plan`, `plan_from_codes` hay `to_digits`.

Yêu cầu:

1. `pigs` bằng đúng `Solution().poorPigs(...)`.
2. Với mọi xô độc, `check_plan(plan, pigs, range(buckets), minutesToDie, minutesToTest)` trả `True`.
3. Thử thách thêm: kế hoạch của bạn **không cho con lợn nào uống ở lượt mà nó chắc chắn đã chết** — thực ra chiến lược cơ số tự động thỏa điều này, hãy giải thích vì sao.

In [33]:
def my_feeding_plan(buckets, minutesToDie, minutesToTest):
    # TODO 1: T, base, pigs (dùng vòng lặp số nguyên)
    # TODO 2: plan = {r: [set() for _ in range(pigs)] for r in range(1, T + 1)}
    # TODO 3: với mỗi xô b, tách chữ số cơ số base bằng % và //;
    #         chữ số d > 0 ở vị trí j -> plan[d][j].add(b)
    pass

# Tự làm xong rồi mới mở test:
# for cfg in [(4, 15, 15), (9, 15, 30), (1000, 15, 60), (1, 1, 1)]:
#     pigs, plan = my_feeding_plan(*cfg)
#     assert pigs == Solution().poorPigs(*cfg)
#     assert check_plan(plan, pigs, range(cfg[0]), cfg[1], cfg[2])[0]

## 21. Gợi ý theo mức

**Bài 1**
- Mức 1: tính `T = minutesToTest // minutesToDie` trước, rồi `states = T + 1`.
- Mức 2: liệt kê `states^0, states^1, ...` cho tới khi `≥ buckets`.
- Mức 3: input 3 có `59 // 12 = 4` lượt (11 phút dư vô dụng). Câu hỏi thêm: lợn 0 sống ⇒ `d₀ = 0`; lợn 1 chết phút 15 ⇒ `d₁ = 1`.

**Bài 2**
- Mức 1: đây chính là bản `Solution` — nhưng hãy viết lại không nhìn.
- Mức 2: `while capacity < buckets:` (dấu `<`).
- Mức 3: trong thân vòng lặp chỉ có `capacity *= states` và `pigs += 1`.

**Bài 3**
- Mức 1: phút chết chia cho `minutesToDie` ra số lượt = chữ số.
- Mức 2: dùng biến `power` bắt đầu bằng 1, mỗi con lợn nhân thêm `base`.
- Mức 3: `digit = 0 if t is None else t // minutesToDie; result += digit * power; power *= base`.

**Bài 4**
- Mức 1: đây là bài "ngược": cố định số lợn, tìm số lượt.
- Mức 2: khi `pigs = 0` thì `(T+1)^0 = 1` với mọi `T` ⇒ vòng lặp sẽ không bao giờ dừng nếu `buckets > 1` — phải xử lý riêng trước.
- Mức 3: `T = 0; while (T + 1) ** pigs < buckets: T += 1`.

**Bài 5**
- Mức 1: mỗi lần cân = một "con lợn" có 3 trạng thái (trái / phải / ngoài).
- Mức 2: `digits = to_digits(c, 3, w)`; `digits[i] == 1` ⇒ cho vào `left` của lần cân `i`.
- Mức 3: giải mã giống hệt `from_digits(digits, 3)`.

**Bài 6**
- Mức 1: sao chép ý của `build_feeding_plan` nhưng tự tách chữ số.
- Mức 2: với mỗi `b`: `n = b; for j in range(pigs): n, d = divmod(n, base); ...`.
- Mức 3 (câu giải thích): lợn `j` chỉ uống xô độc `p` ở **đúng một** lượt `d_j(p)`, nên trước lượt đó nó chắc chắn còn sống; sau lượt đó nó đã chết và simulator bỏ qua — kết quả không đổi.

## 22. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1 (tính tay):**

1. `(27, 10, 30)`: `T = 3`, `states = 4`; `4^2 = 16 < 27`, `4^3 = 64 ≥ 27` ⇒ **3**.
2. `(1000, 15, 15)`: `T = 1`, `states = 2`; `2^9 = 512 < 1000`, `2^10 = 1024` ⇒ **10**.
3. `(25, 12, 59)`: `T = 59 // 12 = 4`, `states = 5`; `5^1 = 5 < 25`, `5^2 = 25 ≥ 25` ⇒ **2** (bằng đúng, dừng vì dấu `<`).
4. `(1, 5, 100)`: một xô ⇒ **0**.

Câu hỏi thêm: `d₀ = 0`, `d₁ = 1` ⇒ xô `0 + 1·3 = 3`. Kiểm tra trong bảng 9 xô ở Cách 3: xô 3 có mã `(0, 1)` ✓.

In [34]:
# Bài 1
answers_1 = [Solution().poorPigs(27, 10, 30), Solution().poorPigs(1000, 15, 15),
             Solution().poorPigs(25, 12, 59), Solution().poorPigs(1, 5, 100)]
assert answers_1 == [3, 10, 2, 0]
assert decode_death_times((None, 15), 15, 3) == 3
print("Bài 1:", answers_1, "| câu thêm: xô 3")


# Bài 2
def answer_poor_pigs_exercise(buckets, minutesToDie, minutesToTest):
    states = minutesToTest // minutesToDie + 1
    pigs = 0
    capacity = 1
    while capacity < buckets:
        capacity *= states
        pigs += 1
    return pigs

assert answer_poor_pigs_exercise(4, 15, 15) == 2
assert answer_poor_pigs_exercise(4, 15, 30) == 2
assert answer_poor_pigs_exercise(1000, 15, 60) == 5
assert answer_poor_pigs_exercise(125, 15, 60) == 3
assert answer_poor_pigs_exercise(1, 1, 1) == 0


# Bài 3
def answer_find_poison(death_minutes, minutesToDie, minutesToTest):
    base = minutesToTest // minutesToDie + 1
    result = 0
    power = 1
    for t in death_minutes:
        digit = 0 if t is None else t // minutesToDie
        result += digit * power
        power *= base
    return result

assert answer_find_poison([15, 30], 15, 30) == 7
assert answer_find_poison([None, None], 15, 30) == 0
assert answer_find_poison([None, 15], 15, 30) == 3
assert answer_find_poison([45, None, 15, 60, 30], 15, 60) == 3 + 1 * 25 + 4 * 125 + 2 * 625
print("Bài 2, Bài 3: PASS")

Bài 1: [3, 10, 2, 0] | câu thêm: xô 3
Bài 2, Bài 3: PASS


In [35]:
# Bài 4
def answer_min_minutes_to_test(buckets, pigs, minutesToDie):
    if buckets == 1:
        return 0
    if pigs == 0:
        return -1
    T = 0
    while (T + 1) ** pigs < buckets:
        T += 1
    return T * minutesToDie

assert answer_min_minutes_to_test(1000, 5, 15) == 45
assert answer_min_minutes_to_test(4, 2, 15) == 15
assert answer_min_minutes_to_test(9, 2, 15) == 30
assert answer_min_minutes_to_test(10, 2, 15) == 45
assert answer_min_minutes_to_test(1, 0, 15) == 0
assert answer_min_minutes_to_test(2, 0, 15) == -1
# đối chiếu: với thời gian vừa tìm được, poorPigs phải <= pigs
for B in range(1, 200, 7):
    for p in range(1, 5):
        t = answer_min_minutes_to_test(B, p, 10)
        assert Solution().poorPigs(B, 10, max(t, 10)) <= p


# Bài 5
def answer_coin_strategy(n):
    w, capacity = 0, 1
    while capacity < n:
        capacity *= 3
        w += 1
    plan = []
    for i in range(w):
        left = {c for c in range(n) if to_digits(c, 3, w)[i] == 1}
        right = {c for c in range(n) if to_digits(c, 3, w)[i] == 2}
        plan.append((left, right))
    return w, plan


def answer_weigh_and_decode(n, plan, heavy):
    digits = []
    for left, right in plan:
        if heavy in left:
            digits.append(1)
        elif heavy in right:
            digits.append(2)
        else:
            digits.append(0)
    return from_digits(digits, 3)

w, plan = answer_coin_strategy(27)
assert w == 3
assert all(answer_weigh_and_decode(27, plan, c) == c for c in range(27))
assert answer_coin_strategy(28)[0] == 4 and answer_coin_strategy(1)[0] == 0
print("Bài 4, Bài 5: PASS")

Bài 4, Bài 5: PASS


In [36]:
# Bài 6
def answer_my_feeding_plan(buckets, minutesToDie, minutesToTest):
    T = minutesToTest // minutesToDie
    base = T + 1
    pigs, capacity = 0, 1
    while capacity < buckets:
        capacity *= base
        pigs += 1
    plan = {r: [set() for _ in range(pigs)] for r in range(1, T + 1)}
    for b in range(buckets):
        n = b
        for j in range(pigs):
            n, d = divmod(n, base)
            if d > 0:
                plan[d][j].add(b)
    return pigs, plan

for cfg in [(4, 15, 15), (9, 15, 30), (1000, 15, 60), (1, 1, 1), (1000, 1, 1)]:
    pigs, plan = answer_my_feeding_plan(*cfg)
    assert pigs == Solution().poorPigs(*cfg)
    assert check_plan(plan, pigs, range(cfg[0]), cfg[1], cfg[2])[0]
print("Bài 6: PASS — chiến lược tự xây giải mã đúng mọi xô")

Bài 6: PASS — chiến lược tự xây giải mã đúng mọi xô


## 23. Template nên nhớ

```python
class Solution:
    def poorPigs(self, buckets: int, minutesToDie: int, minutesToTest: int) -> int:
        states = minutesToTest // minutesToDie + 1   # chết sau lượt 1..T, hoặc sống
        pigs, capacity = 0, 1
        while capacity < buckets:                    # cần states^pigs >= buckets
            capacity *= states
            pigs += 1
        return pigs
```

Câu thần chú: **"Mỗi con lợn là một chữ số cơ số `T + 1`."**

Khuôn chung cho mọi bài "phân biệt N khả năng":

```text
k = số kết quả của một đơn vị thử
x = số nhỏ nhất với k^x >= N   (vòng lặp số nguyên, không dùng log)
Đủ:  mã cơ số k cho từng khả năng
Cần: nguyên lý chuồng bồ câu
```

## 24. Checklist trước khi submit

- Mình đã tính số lượt bằng `//` (chia lấy phần nguyên), không phải `/` chưa?
- Mình đã `+ 1` cho trạng thái "**sống**" chưa?
- Vòng lặp dùng `<` (không phải `<=`) để dừng đúng khi `buckets` là lũy thừa đúng (`4 = 2^2`, `125 = 5^3`)?
- `buckets = 1` có trả `0` không?
- Mình có dùng `math.log` không? Nếu có, đã chỉnh lại bằng số nguyên chưa (thử `(125, 15, 60)` → phải ra `3`)?
- Đã test hai ví dụ của đề `(4, 15, 15) → 2`, `(4, 15, 30) → 2` và `(1000, 15, 60) → 5` chưa?
- Mình có giải thích được **vì sao không thể ít lợn hơn** (đếm kết quả + chuồng bồ câu) không? Đây là câu phỏng vấn viên hay hỏi tiếp.